# Dataset Visualization

This notebook loads a random subset of 16 training images from the strong lensing simulation dataset. The same flux normalization applied during training is used here so the displayed images are representative of what the network sees.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# define your plot style

best_style = {
    "font.family": "sans-serif",
    "mathtext.fontset": "custom",
    "mathtext.rm": "TeX Gyre Heros",
    "mathtext.bf": "TeX Gyre Heros:bold",
    "mathtext.sf": "TeX Gyre Heros",
    "mathtext.it": "TeX Gyre Heros:italic",
    "mathtext.tt": "TeX Gyre Heros",
    "mathtext.cal": "TeX Gyre Heros",
    "mathtext.default": "regular",
    "figure.figsize": (10.0, 10.0),
    "font.size": 26,
    "axes.labelsize": "medium",
    "axes.unicode_minus": False,
    "xtick.labelsize": "small",
    "ytick.labelsize": "small",
    "legend.fontsize": "small",
    "legend.handlelength": 1.5,
    "legend.borderpad": 0.5,
    "xtick.direction": "in",
    "xtick.major.size": 12,
    "xtick.minor.size": 6,
    "xtick.major.pad": 6,
    "xtick.top": True,
    "xtick.major.top": True,
    "xtick.major.bottom": True,
    "xtick.minor.top": True,
    "xtick.minor.bottom": True,
    "xtick.minor.visible": True,
    "ytick.direction": "in",
    "ytick.major.size": 12,
    "ytick.minor.size": 6.0,
    "ytick.right": True,
    "ytick.major.left": True,
    "ytick.major.right": True,
    "ytick.minor.left": True,
    "ytick.minor.right": True,
    "ytick.minor.visible": True,
    "grid.alpha": 0.8,
    "grid.linestyle": ":",
    "axes.linewidth": 2,
    "savefig.transparent": False,
}
plt.style.use(best_style)
cols = ["#5790fc", "#f89c20", "#e42536", "#964a8b", "#9c9ca1", "#7a21dd"]
#set cols as the matplotlib default color cycle
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=cols)

In [ ]:
# dataset directory and image directory
datadir = "region2_narrowprior"

image_dir = f'/deepskieslab/stronglensing/hsbi/datasets/train_data/entire_range_narrowprior_copy1/'
images = np.load(image_dir + '/CONFIGURATION_1_images.npy', allow_pickle=True)
metadata = pd.read_csv(image_dir + '/CONFIGURATION_1_metadata.csv')

In [4]:
len(metadata)

500000

In [ ]:
w_column_name = "w0-g" 
om_column_name = 'Om0-g'
zl_column_name = 'PLANE_1-REDSHIFT-g'
zs_column_name = 'PLANE_2-REDSHIFT-g'
v_column_name = 'PLANE_1-OBJECT_1-MASS_PROFILE_1-sigma_v-g'

In [ ]:
# randomly select 16 images from the dataset for visualization
n_images = 16
np.random.seed(42)
selected_indices = np.random.choice(len(images), n_images, replace=False)

In [ ]:
# Preprocessing of the images
images = images[selected_indices]
images = np.einsum('lkij->lijk', images)
edge_size = images.shape[1]
images = edge_size * edge_size * (images / np.sum(images, axis=(1, 2), keepdims=True))
images = images.reshape(images.shape[0], -1)
images = images.reshape(images.shape[0], edge_size, edge_size, 1)

metadata = metadata.iloc[selected_indices]
aparms = metadata[[zl_column_name,zs_column_name,v_column_name]].to_numpy().astype(np.float32)

In [11]:
print(images.shape, aparms.shape)

(16, 32, 32, 1) (16, 3)


In [10]:
true_w = metadata[w_column_name].unique()
true_om = metadata[om_column_name].unique()
print("True w values in selected images:", true_w)
print("True Omega_m values in selected images:", true_om)

True w values in selected images: [-1.42099496 -0.95426545 -1.23544578 -0.87831851 -0.67819032 -0.69820629
 -1.22467135 -0.54053211 -0.68877798 -0.65434069 -1.40015587 -1.65318424
 -0.46645813 -0.70748638 -0.96456307 -1.12497655]
True Omega_m values in selected images: [0.88379038 0.33263605 0.59810371 0.853615   0.60098733 0.22015494
 0.68581994 0.83073605 0.82674043 0.1426312  0.55632236 0.27157673
 0.89894059 0.44636037 0.1476145  0.32288837]


In [ ]:
# Sort images by w + Om so the grid progresses from low to high combined parameter values,
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
sorted_indices = np.argsort(metadata[w_column_name].values + metadata[om_column_name].values)
for i, ax in enumerate(axes.flat):
    idx = sorted_indices[i]
    img = images[idx, :, :, 0]
    ax.imshow(img, cmap='gray', origin='lower')
    w_val = metadata.iloc[idx][w_column_name]
    om_val = metadata.iloc[idx][om_column_name]
    zl_val = metadata.iloc[idx][zl_column_name]
    zs_val = metadata.iloc[idx][zs_column_name]
    v_val = metadata.iloc[idx][v_column_name]
    ax.text(2.0, 28.0, rf"$w$: {w_val:.2f}, $\Omega_m$: {om_val:.2f}", color='white', fontsize=14)
    ax.axis('off')

# remove gaps between subplots
plt.subplots_adjust(wspace=0.01, hspace=0.01)
#plt.tight_layout()
# plt.show()
plt.savefig("train_data.pdf", dpi=300)